# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Name: {getattr(metadata, 'name', '')}")
print(f"Description: {getattr(metadata, 'description', '')}\n")
print(f"Version: {getattr(metadata, 'version', '')}")
print(f"Published: {getattr(metadata, 'datePublished', '')}")
print(f"License: {getattr(metadata, 'license', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs. This helps you identify what tables and columns are available for exploration.

In [ ]:
# List all record sets by their @id

record_sets = getattr(metadata, 'recordSet', [])
print("Available Record Sets:")
for record_set in record_sets:
    print(f"  @id: {getattr(record_set, '@id', '')} | Name: {getattr(record_set, 'name', '')}")

# For demonstration, print fields for each record set
for record_set in record_sets:
    print(f"\nRecord Set '@id': {getattr(record_set, '@id', '')} - Name: {getattr(record_set, 'name', '')}")
    fields = getattr(record_set, 'field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print(f"  Fields (by @id):")
    for field in fields:
        fname = getattr(field, 'name', '')
        fid = getattr(field, '@id', '')
        fdesc = getattr(field, 'description', '')
        print(f"    - @id: {fid} | Name: {fname} | Description: {fdesc}")

## 3. Data Extraction
Load data from the available record set into a DataFrame for analysis. Use the record set and field `@id`s identified in the overview above.

In [ ]:
# Choose the main record set (table) for analysis
# You can change this to another record set @id if multiple tables are present

if len(record_sets) == 0:
    raise ValueError("No record sets found in the metadata. Check the schema or metadata structure.")

main_record_set = record_sets[0]  # select the first record set as main
main_record_set_id = getattr(main_record_set, '@id', None)

# List all record set @id for completeness
record_set_ids = [getattr(rs, '@id', None) for rs in record_sets]

dataframes = {}
for rid in record_set_ids:
    all_records = list(dataset.records(record_set=rid))
    dataframes[rid] = pd.DataFrame(all_records)

# Display status and available columns for the main record set
print(f"Main Record Set: {main_record_set_id}")
df = dataframes[main_record_set_id]
print(f"DataFrame Columns (@id):\n{df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: Select a numeric field for analysis
# Display all numeric fields for selection

# Try to auto-detect numeric columns
numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric field candidates: {numeric_candidates}")

# If none auto-detected, try guessing by name
if not numeric_candidates:
    numeric_candidates = [c for c in df.columns if ('int' in str(df[c].dtype).lower() or 'float' in str(df[c].dtype).lower() or 'count' in c.lower() or 'abundance' in c.lower() or 'weight' in c.lower() or 'coverage' in c.lower())]
    print(f"Heuristically detected numeric fields: {numeric_candidates}")

# Pick the first numeric field found, otherwise specify manually
if not numeric_candidates:
    raise ValueError("No obvious numeric columns found to analyze. Please inspect the DataFrame.")
numeric_field = numeric_candidates[0]
print(f"Using numeric field: {numeric_field}")

# Drop NA to avoid issues
df_numeric = df.dropna(subset=[numeric_field])

# Set an example threshold at the 90th percentile for demo
threshold = df_numeric[numeric_field].quantile(0.90)
filtered_df = df_numeric[df_numeric[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (top 10%):")
print(filtered_df.head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to group by a categorical field (e.g., 'description', 'sample', etc.)
string_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
print(f"Categorical/group field candidates: {string_candidates}")

# Pick the first available as group
if string_candidates:
    group_field = string_candidates[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (showing mean {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll use matplotlib and seaborn for basic EDA plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.tight_layout()
plt.show()

# Boxplot by group if available
if string_candidates:
    plt.figure(figsize=(12, 4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² Mass Spectrometry dataset using the `mlcroissant` library. We demonstrated how to inspect record sets and fields by their Croissant `@id`, extract and examine data in pandas, apply exploratory data analysis (EDA), and visualize key attributes.

You can further customize your analysis by referencing specific `@id`s for fields and record sets as needed for reproducibility from the Croissant schema.